# F1 Data Cleaning
Cleans `data/processed/final_merged_dataset.csv` and saves the result to `data/processed/cleaned_dataset.csv`.

See `docs/04_data_cleaning.md` for the full writeup, including two upstream bugs found and fixed in the merge pipeline before this notebook was finalized.

## 1. Load data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/final_merged_dataset.csv', low_memory=False)
print('Shape:', df.shape)
df.head()

Shape: (320274, 68)


,raceId,year,round,circuit_name,location,country,date,driverId,driverRef,code,constructor_name,lap,position,grid,milliseconds,cum_time_ms,gap_to_leader_ms,pit_stop_this_lap,cumulative_pit_stops,laps_since_last_pit,pit_duration_ms,statusId,points,positionOrder,won,total_laps,race_progress,position_vs_grid,position_change,rolling_lap_time_ms,pace_delta_to_fastest_ms,gap_to_leader_s,gap_per_lap_covered_s,tire_age_ratio,gap_to_ahead_ms,drs_zone_proxy,pit_window_proxy,driver_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed,pit_stop_duration_s,pit_stop_this_lap_openf1,compound,stint_number,tyre_age_at_stint_start,date_openf1,pressure,track_temperature,rainfall,wind_speed,wind_direction,humidity,air_temperature,meeting_key,session_key,full_name,name_acronym,team_name
0,841,2011,1,Albert Park Grand Prix Circuit,Melbourne,Australia,2011-03-27,20,vettel,VET,Red Bull,1,1,1,98109,98109,0,0,0,1,0.0,1,25.0,1,1,58,0.017241,0,0.0,98109.0,0,0.000,0.000,0.017241,0.0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,841,2011,1,Albert Park Grand Prix Circuit,Melbourne,Australia,2011-03-27,1,hamilton,HAM,McLaren,1,2,2,100573,100573,2464,0,0,1,0.0,1,18.0,2,0,58,0.017241,0,0.0,100573.0,2464,2.464,2.464,0.017241,2464.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,841,2011,1,Albert Park Grand Prix Circuit,Melbourne,Australia,2011-03-27,17,webber,WEB,Red Bull,1,3,3,101467,101467,3358,0,0,1,0.0,1,10.0,5,0,58,0.017241,0,0.0,101467.0,3358,3.358,3.358,0.017241,894.0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,841,2011,1,Albert Park Grand Prix Circuit,Melbourne,Australia,2011-03-27,808,petrov,PET,Renault,1,4,6,102835,102835,4726,0,0,1,0.0,1,15.0,3,0,58,0.017241,2,0.0,102835.0,4726,4.726,4.726,0.017241,1368.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,841,2011,1,Albert Park Grand Prix Circuit,Melbourne,Australia,2011-03-27,13,massa,MAS,Ferrari,1,5,8,104196,104196,6087,0,0,1,0.0,1,6.0,7,0,58,0.017241,3,0.0,104196.0,6087,6.087,6.087,0.017241,1361.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Core identifier check
These columns should never be null — if they are, something broke upstream in the merge pipeline, not something to fix here.

In [2]:
core_cols = ['raceId', 'driverId', 'lap', 'position', 'grid', 'year', 'won']
print(df[core_cols].isnull().sum())

raceId      0
driverId    0
lap         0
position    0
grid        0
year        0
won         0
dtype: int64


## 3. Row-level duplicate check
Each (race, driver, lap) combination should be unique.

In [3]:
dupe_key = ['raceId', 'driverId', 'lap']
dupes = df.duplicated(subset=dupe_key, keep=False)
print(f'Duplicate rows on {dupe_key}: {dupes.sum()}')

if dupes.sum() > 0:
    print(df[dupes].sort_values(dupe_key).head(10))
    df = df.drop_duplicates(subset=dupe_key, keep='first')
    print('Dropped duplicates. New shape:', df.shape)

Duplicate rows on ['raceId', 'driverId', 'lap']: 0


## 4. Cross-source validation
Two independent measurements of the same real-world events (pit stops, driver identity) should agree. This both validates data quality AND identifies which of two duplicate columns to keep.

In [4]:
both_present = df[df['session_key'].notna()][['pit_stop_this_lap', 'pit_stop_this_lap_openf1']]
agreement_pct = (both_present['pit_stop_this_lap'] == both_present['pit_stop_this_lap_openf1']).mean() * 100
print(f'Pit stop agreement: {agreement_pct:.1f}% ({len(both_present)} rows compared)')
print('Note: ~1.1% disagreement traced to 7/22 races where OpenF1 pit endpoint had no data')
print('(see docs/04_data_cleaning.md) — not a pipeline bug, just an upstream gap.')

Pit stop agreement: 98.9% (23431 rows compared)
Note: ~1.1% disagreement traced to 7/22 races where OpenF1 pit endpoint had no data
(see docs/04_data_cleaning.md) — not a pipeline bug, just an upstream gap.


In [5]:
both_present2 = df[df['session_key'].notna()][['code', 'name_acronym']]
agreement_pct2 = (both_present2['code'] == both_present2['name_acronym']).mean() * 100
print(f'Driver code agreement: {agreement_pct2:.1f}% ({len(both_present2)} rows compared)')

Driver code agreement: 100.0% (23431 rows compared)


## 5. Drop redundant columns
Now that cross-validation above confirms these sources agree, keep the Ergast versions (they cover all years) and drop the OpenF1 duplicates.

In [6]:
cols_to_drop = ['pit_stop_this_lap_openf1', 'full_name', 'name_acronym', 'team_name']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print('Columns now:', df.shape[1])

Columns now: 64


## 6. Handle missing values

In [7]:
pit_cols = ['pit_stop_this_lap', 'pit_stop_duration_s', 'cumulative_pit_stops', 'pit_duration_ms']
for col in pit_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [8]:
openf1_cols = [c for c in ['compound', 'air_temperature', 'track_temperature', 'rainfall'] if c in df.columns]
df['has_openf1_telemetry'] = df[openf1_cols].notna().any(axis=1).astype(int)
print('Rows with real OpenF1 telemetry:', df['has_openf1_telemetry'].sum())
print('Rows without (Ergast-only):', (df['has_openf1_telemetry'] == 0).sum())

Rows with real OpenF1 telemetry: 23431
Rows without (Ergast-only): 296843


## 7. Data type enforcement

In [ ]:
int_cols = ['raceId', 'driverId', 'lap', 'position', 'grid', 'year', 'won', 'pit_stop_this_lap']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
print(df[[c for c in int_cols if c in df.columns]].dtypes)

raceId               Int64
driverId             Int64
lap                  Int64
position             Int64
grid                 Int64
year                 Int64
won                  Int64
pit_stop_this_lap    Int64
dtype: object


## 8. Anomalous lap time flag
Laps over 10 minutes are red-flag/safety-car artifacts (confirmed via statusId and multi-driver clustering — see docs), not data errors. Flag rather than drop, so the driver's still-valid result isn't lost.

In [10]:
df['anomalous_lap_time'] = (df['milliseconds'] / 1000 > 600).astype(int)
print('Flagged anomalous laps:', df['anomalous_lap_time'].sum())

Flagged anomalous laps: 562


## 9. Final check and save

In [11]:
print('Final shape:', df.shape)
print()
print('Remaining nulls (top 15):')
print(df.isnull().sum().sort_values(ascending=False).head(15))

Final shape: (320274, 66)

Remaining nulls (top 15):
i1_speed             300424
st_speed             298848
wind_direction       298228
wind_speed           298228
rainfall             298228
track_temperature    298228
air_temperature      298228
humidity             298228
pressure             298228
date_openf1          298228
duration_sector_1    296972
duration_sector_3    296907
date_start           296889
lap_duration         296861
duration_sector_2    296844
dtype: int64


In [12]:
df.to_csv('../data/processed/cleaned_dataset.csv', index=False)
print('Saved -> data/processed/cleaned_dataset.csv')
print('Final shape:', df.shape)

Saved -> data/processed/cleaned_dataset.csv
Final shape: (320274, 66)


In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/cleaned_dataset.csv', low_memory=False)

print(df['pit_duration_ms'].describe())
print()
print("Median (a more robust 'typical' value):", df.loc[df['pit_duration_ms'] > 0, 'pit_duration_ms'].median())

count    3.202740e+05
mean     3.026021e+03
std      6.057893e+04
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.069017e+06
Name: pit_duration_ms, dtype: float64

Median (a more robust 'typical' value): 23606.0
